# PP12: классификатор строительных задач

Цель notebook - обучить небольшую ML-модель, которая по короткому названию задачи определяет категорию работ. Категория затем используется сервисом PP12 для улучшения prompt к LM Studio и подбора похожих примеров.

## 1. Импорт библиотек

In [ ]:
from pathlib import Path
from io import StringIO
import re

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
plt.style.use("seaborn-v0_8-whitegrid")

## 2. Загрузка датасета

In [ ]:
def find_project_root() -> Path:
    """Find pp12_ml root when notebook is launched from repo root, pp12_ml, or notebooks."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        direct_dataset = candidate / "data" / "construction_tasks_dataset.csv"
        nested_dataset = candidate / "pp12_ml" / "data" / "construction_tasks_dataset.csv"
        if direct_dataset.exists():
            return candidate
        if nested_dataset.exists():
            return candidate / "pp12_ml"
    raise FileNotFoundError("Could not find pp12_ml/data/construction_tasks_dataset.csv")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "construction_tasks_dataset.csv"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

df = pd.read_csv(DATA_PATH)
df.head()

## 3. Первичный анализ данных

In [ ]:
print("Shape:", df.shape)
display(df.head(10))

buffer = StringIO()
df.info(buf=buffer)
print(buffer.getvalue())

print("Пропуски по столбцам:")
display(df.isna().sum())

print("Дубликаты по title:", df.duplicated(subset=["title"]).sum())
print("Дубликаты полных строк:", df.duplicated().sum())

print("Распределение по категориям:")
display(df["category"].value_counts())

## 4. Визуализация

In [ ]:
df["title_len"] = df["title"].astype(str).str.len()
df["description_len"] = df["reference_description"].astype(str).str.len()

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

df["category"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="#4c78a8")
axes[0].set_title("Распределение категорий")
axes[0].set_xlabel("Категория")
axes[0].set_ylabel("Количество")
axes[0].tick_params(axis="x", rotation=45)

df["title_len"].plot(kind="hist", bins=15, ax=axes[1], color="#59a14f")
axes[1].set_title("Длина названий задач")
axes[1].set_xlabel("Символы")

df["description_len"].plot(kind="hist", bins=15, ax=axes[2], color="#f28e2b")
axes[2].set_title("Длина эталонных описаний")
axes[2].set_xlabel("Символы")

plt.tight_layout()
plt.show()

## 5. Предобработка текста

In [ ]:
def preprocess_text(value: str) -> str:
    """Базовая нормализация русскоязычных коротких названий задач."""
    text = str(value or "").lower().replace("ё", "е")
    text = re.sub(r"[^0-9a-zа-я\s-]", " ", text)
    text = re.sub(r"[-_]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["title_clean"] = df["title"].apply(preprocess_text)
df[["title", "title_clean", "category"]].head(10)

## 6. Подготовка целевой переменной

In [ ]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df["category"])

label_mapping = pd.DataFrame({
    "class_id": range(len(label_encoder.classes_)),
    "category": label_encoder.classes_,
})
display(label_mapping)

## 7. Разделение выборки 70/15/15

In [ ]:
X_raw = df["title_clean"]

X_train_val_raw, X_test_raw, y_train_val, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.15,
    random_state=RANDOM_STATE,
    stratify=y,
)

validation_size_from_train_val = 0.15 / 0.85
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_train_val_raw,
    y_train_val,
    test_size=validation_size_from_train_val,
    random_state=RANDOM_STATE,
    stratify=y_train_val,
)

print("Train:", len(X_train_raw))
print("Validation:", len(X_val_raw))
print("Test:", len(X_test_raw))

## 8. Подготовка признаков TF-IDF

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    max_features=3000,
)

X_train = tfidf_vectorizer.fit_transform(X_train_raw)
X_val = tfidf_vectorizer.transform(X_val_raw)
X_test = tfidf_vectorizer.transform(X_test_raw)

print("TF-IDF train shape:", X_train.shape)

## 9. Обучение моделей

In [ ]:
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "LinearSVC": LinearSVC(
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
}

trained_models = {}
for name, model in models.items():
    trained_models[name] = model.fit(X_train, y_train)
    print(f"Обучена модель: {name}")

## 10. Оценка моделей на validation

In [ ]:
def evaluate_model(model, X, y_true):
    y_pred = model.predict(X)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

metrics_rows = []
for name, model in trained_models.items():
    metrics = evaluate_model(model, X_val, y_val)
    metrics_rows.append({"model": name, **metrics})
    print(f"\n{name}")
    print(classification_report(
        y_val,
        model.predict(X_val),
        target_names=label_encoder.classes_,
        zero_division=0,
    ))

comparison = pd.DataFrame(metrics_rows).sort_values("f1_macro", ascending=False)
comparison

## 11. Выбор лучшей модели

In [ ]:
best_model_name = comparison.iloc[0]["model"]
best_model = trained_models[best_model_name]

print("Лучшая модель по f1_macro:", best_model_name)
display(comparison)

## 12. Переобучение лучшей модели на train + validation

In [ ]:
X_train_final_raw = pd.concat([X_train_raw, X_val_raw])
y_train_final = list(y_train) + list(y_val)

final_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    max_features=3000,
)
X_train_final = final_vectorizer.fit_transform(X_train_final_raw)
X_test_final = final_vectorizer.transform(X_test_raw)

final_model = clone(models[best_model_name])
final_model.fit(X_train_final, y_train_final)

print("Финальная модель обучена:", best_model_name)

## 13. Финальная оценка на test

In [ ]:
test_pred = final_model.predict(X_test_final)
test_metrics = evaluate_model(final_model, X_test_final, y_test)

display(pd.DataFrame([{ "model": best_model_name, **test_metrics }]))
print(classification_report(
    y_test,
    test_pred,
    target_names=label_encoder.classes_,
    zero_division=0,
))

## 14. Матрица ошибок

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_pred,
    display_labels=label_encoder.classes_,
    xticks_rotation=45,
    cmap="Blues",
    ax=ax,
)
ax.set_title("Матрица ошибок на test")
plt.tight_layout()
plt.show()

## 15. Примеры предсказаний

In [ ]:
sample_titles = [
    "Заливка ростверка",
    "Монтаж оконных блоков",
    "Шпаклевка стен",
    "Прокладка кабеля",
    "Заказ бетона",
    "Согласование графика",
]

sample_clean = [preprocess_text(title) for title in sample_titles]
sample_features = final_vectorizer.transform(sample_clean)
sample_pred = final_model.predict(sample_features)

pd.DataFrame({
    "title": sample_titles,
    "title_clean": sample_clean,
    "predicted_category": label_encoder.inverse_transform(sample_pred),
})

## 16. Сохранение артефактов

In [ ]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(final_model, ARTIFACTS_DIR / "best_model.joblib")
joblib.dump(final_vectorizer, ARTIFACTS_DIR / "tfidf_vectorizer.joblib")
joblib.dump(label_encoder, ARTIFACTS_DIR / "label_encoder.joblib")

print("Артефакты сохранены в:", ARTIFACTS_DIR.resolve())
print("Модель:", best_model_name)